# 06 — Topology fingerprints on real data (inspection run)

Computes the v0 `TopologyFingerprint` for every hospital, on the **full ARMD cohort**,
for five splits — then prints similarity matrices + aggregation weights, and saves a
compact JSON you paste back to chat for analysis.

**Before running (important — read this):**
1. Runtime → **CPU is enough** (no torch/GPU needed; this module is pure pandas/numpy).
2. `topology.py` is **not pushed to GitHub yet**, so the clone below won't have it.
   After cell 1 finishes, upload YOUR copy of `src/amr_fed/topology.py` from your Mac
   into Colab at `/content/Capstone-amr-fed/src/amr_fed/topology.py`
   (left sidebar → Files → navigate → right-click folder → Upload).
   Cell 3 checks for it and stops loudly if it's missing.
3. Open this notebook itself via **File → Upload notebook** (it isn't pushed either).
4. Total runtime ~10–25 min (one full-cohort load + ~20 hospital fingerprints).


In [ ]:
# 1) Get the code (clone main, or pull if already cloned)
!git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
!pip install -q networkx   # needed only for the Louvain split + optional community feature

## Setup — data + imports

In [ ]:
# 2) Point at the data (mount Drive, set ARMD_DIR before importing amr_fed)
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # EDIT to your ARMD folder

In [ ]:
# 3) Imports + check that topology.py was uploaded (see the header instructions)
import sys
from pathlib import Path
REPO = Path('/content/Capstone-amr-fed')
sys.path.insert(0, str(REPO / 'src'))
topo_path = REPO / 'src' / 'amr_fed' / 'topology.py'
assert topo_path.exists(), (
    'STOP: src/amr_fed/topology.py is missing — upload it from your Mac '
    'into /content/Capstone-amr-fed/src/amr_fed/ (see header), then re-run from here.'
)
from amr_fed import config
from amr_fed.data_loader import load_cohort_frame, PK, ORG, ABX
from amr_fed.partition import (
    dirichlet_ward_mixture, specimen_baseline, organism_community,
    homophily_split, louvain_split,
)
from amr_fed.topology import (
    FEATURE_NAMES, compute_fingerprint, fingerprint_matrix, zscore,
    cosine_similarity_matrix, distance_to_centroid, aggregation_weights,
)
import numpy as np, pandas as pd, json
print('topology features:', FEATURE_NAMES)

## Load the full cohort

In [ ]:
# 4) Full 1.60M-row frame (~2-4 min)
df = load_cohort_frame()
print('rows:', f'{len(df):,}', '| patients:', f'{df[PK].nunique():,}',
      '| label rate:', round(float(df["label"].mean()), 4))

## Helper — fingerprint every hospital of one split

In [ ]:
def fingerprint_hospitals(df, assignment, name):
    """assignment: Series patient -> hospital label (any label type).
    Prints a hospital table + fingerprint matrix + cosine heatmap data + weights.
    Returns a JSON-serialisable dict."""
    mapped = df[PK].map(assignment)
    assert mapped.notna().all(), 'some patients have no hospital assignment!'
    labels = sorted(mapped.unique().tolist(), key=str)
    rows, fps = [], []
    for lab in labels:
        sub = df[mapped == lab]
        fp = compute_fingerprint(sub)
        fps.append(fp)
        rows.append({
            'hospital': str(lab),
            'n_patients': int(sub[PK].nunique()),
            'n_tests': int(len(sub)),
            'resistance_rate': round(float(sub['label'].mean()), 4),
        })
    F = fingerprint_matrix(fps)
    S = cosine_similarity_matrix(zscore(F))
    dist = distance_to_centroid(F)
    out = {
        'split': name,
        'hospitals': rows,
        'fingerprints': {str(lab): [round(float(v), 4) for v in fp.features]
                         for lab, fp in zip(labels, fps)},
        'cosine_similarity': [[round(float(v), 4) for v in row] for row in S],
        'distance_to_centroid': [round(float(v), 4) for v in dist],
        'weights': {},
    }
    for mode in ('consensus', 'distinctiveness'):
        for T in (0.5, 1.0, float('inf')):
            w = aggregation_weights(fps, temperature=T, mode=mode)
            out['weights'][f'{mode}/T={T}'] = [round(float(v), 4) for v in w]
    print(f'--- {name} ---')
    print(pd.DataFrame(rows).to_string(index=False))
    print()
    print('fingerprints (hospitals x 14 features):')
    print(pd.DataFrame(out['fingerprints'],
                       index=list(FEATURE_NAMES)).round(3).to_string())
    print()
    print('pairwise cosine (z-scored):')
    print(pd.DataFrame(out['cosine_similarity'],
                       index=[str(l) for l in labels],
                       columns=[str(l) for l in labels]).round(3).to_string())
    print()
    print('distance_to_centroid:', out['distance_to_centroid'])
    print('weights:', {k: v for k, v in out['weights'].items()})
    return out


results = {}

## Run the splits (one cell each, so a failure doesn't lose everything)

In [ ]:
# 5) Ward-Dirichlet alpha=0.5, 5 hospitals (the standard baseline split)
a = dirichlet_ward_mixture(df, n_clients=5, alpha=0.5, seed=42)
results['ward-dirichlet-0.5'] = fingerprint_hospitals(
    df, a, 'ward-dirichlet alpha=0.5 (5 hospitals)')

In [ ]:
# 6) Specimen split (natural: urine / respiratory / blood / ...)
b = specimen_baseline(df)
results['specimen'] = fingerprint_hospitals(df, b, 'specimen (natural)')

In [ ]:
# 7) Organism-community split (5 hospitals, disjoint bug sets)
c = organism_community(df, n_clients=5, seed=42)
results['organism-community'] = fingerprint_hospitals(
    df, c, 'organism-community (5 hospitals)')

In [ ]:
# 8) Homophily spectrum (4 hospitals) + Louvain communities (5 hospitals)
h = homophily_split(df, n_clients=4, seed=42)
results['homophily'] = fingerprint_hospitals(df, h, 'homophily spectrum (4 hospitals)')
lv = louvain_split(df, n_clients=5, seed=42)
results['louvain'] = fingerprint_hospitals(df, lv, 'louvain communities')

## Visuals — similarity heatmaps

In [ ]:
import matplotlib.pyplot as plt
names = list(results)
fig, axes = plt.subplots(1, len(names), figsize=(4 * len(names), 4.2))
if len(names) == 1:
    axes = [axes]
for ax, name in zip(axes, names):
    S = np.array(results[name]['cosine_similarity'])
    labs = [h['hospital'] for h in results[name]['hospitals']]
    im = ax.imshow(S, vmin=-1, vmax=1, cmap='RdYlBu')
    ax.set_title(name, fontsize=9)
    ax.set_xticks(range(len(S))); ax.set_yticks(range(len(S)))
    ax.set_xticklabels(labs, fontsize=7); ax.set_yticklabels(labs, fontsize=7)
fig.colorbar(im, ax=list(axes), label='cosine similarity (z-scored fingerprints)')
plt.tight_layout(); plt.show()

## Save + report back

Writes a compact JSON to Drive **and** prints it below.
**Copy everything between the COPY lines and paste it back into chat**
(or upload the file if it's too long).

In [ ]:
payload = {
    'feature_names': list(FEATURE_NAMES),
    'dataset': {'rows': int(len(df)), 'patients': int(df[PK].nunique()),
                'label_rate': round(float(df['label'].mean()), 4)},
    'splits': results,
}
# sanity: every weight vector sums to 1, T=inf is uniform
# (tolerance 1e-3: the stored weights are rounded to 4dp, and k-way rounding can
# shift a sum by up to k*5e-5 — the module itself is exact, see test_topology.py)
for name, r in results.items():
    k = len(r['hospitals'])
    for wname, w in r['weights'].items():
        assert abs(sum(w) - 1.0) < 1e-3, (name, wname)
        if wname.endswith('T=inf'):
            assert all(abs(v - 1.0 / k) < 1e-3 for v in w), (name, wname)
print('sanity checks passed (weights sum to 1, T=inf uniform)')
path = '/content/drive/MyDrive/amr_topology_fingerprints.json'
with open(path, 'w') as f:
    json.dump(payload, f)
print('saved', path)
print('=== COPY EVERYTHING BELOW THIS LINE BACK TO CHAT ===')
print(json.dumps(payload))
print('=== END ===')